# Chapter 13 — Timeseries Forecasting (RNNs)

Maps to Chollet Ch.13. A **timeseries** is values measured at regular intervals; **forecasting** = predict
what comes next. Here, order and memory matter — a new ingredient vs. everything so far.

### The workflow (memorize)
1. **Normalize** each feature using **train statistics only**.
2. **Split by time**: train = oldest, val/test = most recent (never shuffle across the split — you predict the
   future from the past, not vice-versa).
3. **Window** the series into `(sequence, target_ahead)` pairs with `timeseries_dataset_from_array`.
4. **Set a commonsense baseline** (predict "same as now"). It's often *hard* to beat.
5. Try Dense → Conv1D → **RNN/LSTM**, beating the baseline.

We use a fast **synthetic** series so everything runs offline; the real **Jena weather** pipeline is in the
templates.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, matplotlib.pyplot as plt
from keras import layers


## 1. A synthetic multivariate timeseries
Temperature with a strong **daily cycle** (period 144 = one day at 10-min sampling), a slow drift, and noise,
plus a correlated pressure channel. We'll forecast **72 steps (half a day) ahead** — far enough that "predict
now" is a weak baseline.

In [ ]:
rng = np.random.default_rng(0); N = 24000; t = np.arange(N)
daily = np.sin(2*np.pi*t/144); slow = np.sin(2*np.pi*t/2000)
temp  = (10 + 5*daily + 2*slow + rng.normal(0, 0.7, N)).astype("float32")
press = (1000 + 3*slow + rng.normal(0, 1.0, N)).astype("float32")
raw   = np.stack([temp, press], axis=1)                       # (N, 2 features)

n_train = int(0.5*N); n_val = int(0.25*N)
# normalize features on TRAIN stats only
mean = raw[:n_train].mean(0); std = raw[:n_train].std(0)
raw_n = (raw - mean) / std
# target = temperature, normalized (we'll report MAE back in °C via *t_std)
t_mean, t_std = temp[:n_train].mean(), temp[:n_train].std()
target = (temp - t_mean) / t_std

plt.plot(temp[:1440]); plt.title("temperature, first 10 days"); plt.xlabel("10-min steps"); plt.show()


## 2. Windowing with `timeseries_dataset_from_array`
It slides a window of `sequence_length` over the series and pairs it with a target `horizon` steps later — no
manual loops, no wasted memory. We build train/val/test from disjoint **time** ranges.

In [ ]:
SEQ_LEN, HORIZON, BATCH = 96, 72, 128
def make_ds(start, end=None):
    return keras.utils.timeseries_dataset_from_array(
        raw_n[:-HORIZON], targets=target[HORIZON:],
        sequence_length=SEQ_LEN, batch_size=BATCH,
        start_index=start, end_index=end, shuffle=True)

train_ds = make_ds(0, n_train)
val_ds   = make_ds(n_train, n_train + n_val)
test_ds  = make_ds(n_train + n_val)

for x, y in train_ds:
    print("samples:", x.shape, " targets:", y.shape); break    # (128, 96, 2) (128,)


## 3. Commonsense baseline: "the future = now"
Predict the last observed temperature. Report **MAE in °C**. Any ML model must beat this to earn its keep.

In [ ]:
def naive_mae(ds):
    err = 0.0; n = 0
    for x, y in ds:
        pred = x[:, -1, 0].numpy()                  # last normalized temperature
        err += np.sum(np.abs((pred - y.numpy()) * t_std))   # back to °C
        n += len(y)
    return err / n
print(f"baseline (persistence) val MAE = {naive_mae(val_ds):.2f} °C")


## 4. Try models: Dense → Conv1D → LSTM
Loss = **MSE** (smooth for gradients); we monitor **MAE**. We report test MAE in °C (`metric * t_std`).

In [ ]:
F = raw.shape[-1]
def evaluate(model, name, epochs=5):
    model.compile("adam", "mse", metrics=["mae"])
    model.fit(train_ds, epochs=epochs, validation_data=val_ds, verbose=0)
    mae_c = model.evaluate(test_ds, verbose=0, return_dict=True)["mae"] * t_std
    print(f"{name:8s} test MAE = {mae_c:.2f} °C")
    return mae_c

# Dense: flattens the window -> no notion of time order
dense = keras.Sequential([keras.Input((SEQ_LEN, F)), layers.Flatten(),
                          layers.Dense(16, activation="relu"), layers.Dense(1)])
# Conv1D: slides 1D windows; pooling discards some order info
conv = keras.Sequential([keras.Input((SEQ_LEN, F)),
                         layers.Conv1D(8, 12, activation="relu"), layers.MaxPooling1D(2),
                         layers.Conv1D(8, 6, activation="relu"),
                         layers.GlobalAveragePooling1D(), layers.Dense(1)])
# LSTM: processes the sequence step by step, keeping memory
lstm = keras.Sequential([keras.Input((SEQ_LEN, F)), layers.LSTM(16), layers.Dense(1)])

for m, name in [(dense,"Dense"), (conv,"Conv1D"), (lstm,"LSTM")]:
    evaluate(m, name)


**Reading the result honestly.** All three beat the persistence baseline here. On this *clean, globally
periodic* series, even the Dense net does great — long-range memory isn't essential, because the whole
flattened window already contains the phase. The LSTM's real advantage shows on **messy real-world series**
(noise, irregular dynamics) where:
- **Dense** flattens away time order and overfits noise,
- **Conv1D**'s pooling destroys ordering and assumes translation-invariance (false for weather: morning ≠ night),
- **LSTM** keeps a running memory and respects causality → it wins.

Verify this yourself on the real **Jena** dataset (template T4): there the LSTM beats Dense/Conv1D, matching
the book (~2.5 °C test MAE vs a 2.6 °C baseline).

## 5. How an RNN works — from scratch
An RNN keeps a **hidden state** `h` and updates it at every timestep using the input *and* the previous state:
$$h_t = \tanh(x_t W_x + h_{t-1} W_h + b)$$
That recurrence is the "memory." The final `h_T` summarizes the whole sequence. Below: a NumPy forward pass,
verified to match Keras's `SimpleRNN` exactly.

In [ ]:
def simple_rnn_forward(x, W_x, W_h, b):
    # x: (batch, timesteps, features)
    B, T, _ = x.shape
    units = W_x.shape[1]
    h = np.zeros((B, units), dtype="float32")
    for step in range(T):
        h = np.tanh(x[:, step, :] @ W_x + h @ W_h + b)   # the recurrence
    return h                                              # last hidden state

# verify against Keras
xseq = np.random.RandomState(1).randn(4, 10, F).astype("float32")
rnn_layer = layers.SimpleRNN(8); _ = rnn_layer(xseq)     # build -> create weights
W_x, W_h, b = rnn_layer.get_weights()
diff = np.abs(simple_rnn_forward(xseq, W_x, W_h, b) - rnn_layer(xseq).numpy()).max()
print(f"max |from-scratch RNN - Keras| = {diff:.2e}  (exact match)")


### Why LSTM/GRU instead of a plain RNN?
A simple RNN forgets quickly — over long sequences gradients **vanish** (Ch.9's telephone game again). The
**LSTM** adds a **cell state** plus **input/forget/output gates** that learn what to keep, forget, and emit,
carrying information across many steps. The **GRU** is a lighter variant (fewer gates), often as good and
faster. In Keras they're drop-in: `layers.LSTM(units)` / `layers.GRU(units)`.

## 6. Making RNNs stronger (the levers)
- **Stack** RNN layers: all but the last need `return_sequences=True` (pass full sequences upward).
- **Recurrent dropout** (`dropout=`, `recurrent_dropout=`) to fight overfitting.
- **Bidirectional** (`layers.Bidirectional(LSTM(...))`) reads the sequence both ways — great for text/
  classification (not for causal forecasting, where the future isn't available).

In [ ]:
stacked = keras.Sequential([
    keras.Input((SEQ_LEN, F)),
    layers.GRU(16, recurrent_dropout=0.0, return_sequences=True),  # pass sequences up
    layers.GRU(16),                                                # last RNN -> final vector
    layers.Dropout(0.3),
    layers.Dense(1),
])
evaluate(stacked, "GRU x2")


---
# ✍️ PROBLEMS

### P1 — Forecast horizon sweep
Re-run the baseline and the LSTM for `HORIZON ∈ {6, 36, 72, 144}`. Plot MAE vs horizon for both. At which
horizon does persistence stop being a strong baseline, and why?

In [ ]:
# TODO


### P2 — From-scratch RNN, extended
Extend `simple_rnn_forward` to also return **all** hidden states (like `return_sequences=True`) and verify
against `layers.SimpleRNN(8, return_sequences=True)`. Then plot how one hidden unit evolves over a sequence.

In [ ]:
# TODO


### P3 — Stacking & dropout
On the synthetic data, compare: single LSTM(16), stacked LSTM(16)x2, and stacked + `recurrent_dropout=0.3`.
Which generalizes best (val vs train MAE gap)?

In [ ]:
# TODO


### P4 — Real data (Jena)
Use template T4 to download the Jena weather set. Reproduce: baseline ≈2.6°C, Dense and Conv1D struggling,
LSTM ≈2.5°C. Confirm the LSTM beats the others here (unlike on the easy synthetic series).

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Windowing

In [ ]:
import keras
# normalize on train stats only:  raw_n = (raw - raw[:n_train].mean(0)) / raw[:n_train].std(0)
ds = keras.utils.timeseries_dataset_from_array(
    raw_n[:-HORIZON], targets=target[HORIZON:],
    sequence_length=SEQ_LEN, batch_size=128,
    start_index=START, end_index=END, shuffle=True)   # split by time, never across


### T2 — RNN/LSTM forecaster

In [ ]:
from keras import layers
model = keras.Sequential([
    keras.Input((SEQ_LEN, NUM_FEATURES)),
    layers.LSTM(32, return_sequences=True),     # stack: all but last return sequences
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(1),                            # regression -> no activation
])
model.compile("adam", "mse", metrics=["mae"])   # forecasting = regression


### T3 — Commonsense baseline (always compute first)

In [ ]:
def naive_mae(ds, feat_col, mean, std):
    err=0.0; n=0
    for x,y in ds:
        pred = x[:, -1, feat_col].numpy()*std + mean    # 'same as now', un-normalized
        err += float(np.sum(np.abs(pred - y.numpy()))); n += len(y)
    return err/n


### T4 — Real Jena weather dataset (Colab)

In [ ]:
# !wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip && unzip -o jena_climate_2009_2016.csv.zip
# import numpy as np
# lines = open("jena_climate_2009_2016.csv").read().split("\n")[1:]
# raw = np.array([[float(v) for v in l.split(",")[1:]] for l in lines if l])   # 14 features
# temperature = raw[:, 1]                                  # column 1 = T (degC)
# n_train=int(0.5*len(raw)); mean=raw[:n_train].mean(0); std=raw[:n_train].std(0); raw=(raw-mean)/std
# sampling_rate=6; sequence_length=120; delay=sampling_rate*(sequence_length+24-1)
# train = keras.utils.timeseries_dataset_from_array(raw[:-delay], targets=temperature[delay:],
#         sampling_rate=sampling_rate, sequence_length=sequence_length, batch_size=256,
#         start_index=0, end_index=n_train, shuffle=True)   # build val/test similarly by time


---
### ✅ Checklist
- [ ] Normalize on train stats only; split timeseries by time (no shuffling across the split).
- [ ] Window data with `timeseries_dataset_from_array` (sequence + target `HORIZON` ahead).
- [ ] Always set the persistence baseline and beat it.
- [ ] Explain why Dense (loses order) and Conv1D (pooling kills order) can struggle on real series.
- [ ] Write the RNN recurrence `h_t = tanh(x_t W_x + h_{t-1} W_h + b)` from scratch and match Keras.
- [ ] Know LSTM/GRU gates fix vanishing gradients; use return_sequences to stack; Bidirectional for non-causal tasks.

🎉 **That completes the Ch.7–14 syllabus** (7, 8, 9, 10, 11, 12, 13, 14) plus the Ch.1–5 foundations.
For the lab final, the from-scratch pieces are now all in hand: perceptron & MLP backprop (Ch.4),
convolution & pooling (Ch.8), the RNN cell (Ch.13), plus IoU/NMS and text/BoW from scratch. If you want, I
can assemble a **timed mock contest** mixing tabular / image / text / timeseries tasks. Say the word.